In [ ]:
# ============================================================
# 03B_dim_label_local.ipynb
# Mục tiêu:
# - Read unresolved content exported from Databricks
# - Apply exact rules again (optional)
# - LLM classify remaining unresolved content
# - Save dim_content_new_labels.parquet / csv
# ============================================================


# CONFIG
import os
import re
import json
import math
import unicodedata
import pandas as pd
import requests
from typing import Optional

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

PATH_UNRESOLVED = "/content/dim_content_unresolved.parquet"
PATH_RULES = "/content/dim_content_rules.parquet"

OUTPUT_DIR = "/content/customer360_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_OUTPUT_PARQUET = os.path.join(OUTPUT_DIR, "dim_content_new_labels.parquet")
PATH_OUTPUT_CSV = os.path.join(OUTPUT_DIR, "dim_content_new_labels.csv")
PATH_CHECKPOINT = os.path.join(OUTPUT_DIR, "llm_checkpoint.jsonl")

OLLAMA_HOST = "https://oxidation-cursive-property.ngrok-free.dev"
OLLAMA_MODEL = "qwen2.5:7b"
LLM_BATCH_SIZE = 20
LLM_TIMEOUT = 180

FIXED_CATEGORIES = [
    "C-Drama",
    "K-Drama",
    "V-Drama",
    "Thai-Drama",
    "Action",
    "Animation",
    "Music",
    "Sport",
    "Reality Show",
    "Documentary",
    "Other"
]

print("Output dir:", OUTPUT_DIR)

Output dir: /content/customer360_output


In [ ]:
# HELPERS - IO
def read_table_file(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    path_lower = path.lower()
    if path_lower.endswith(".parquet"):
        return pd.read_parquet(path)
    elif path_lower.endswith(".csv"):
        return pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported format: {path}")

In [ ]:
# HELPERS - TEXT NORMALIZATION]
def normalize_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None

    text = str(text).strip().lower()
    if not text:
        return None

    text = re.sub(r"[\"'`]+", "", text)
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return None

    text_no_accent = unicodedata.normalize("NFD", text)
    text_no_accent = "".join(
        ch for ch in text_no_accent if unicodedata.category(ch) != "Mn"
    )
    text_no_accent = re.sub(r"\s+", " ", text_no_accent).strip()

    return text_no_accent if text_no_accent else None

In [ ]:
# HELPERS - CATEGORY
def normalize_category(cat: Optional[str]) -> str:
    if cat is None:
        return "Other"

    cat = str(cat).strip()
    for c in FIXED_CATEGORIES:
        if c.lower() == cat.lower():
            return c

    for c in FIXED_CATEGORIES:
        if c.lower() in cat.lower():
            return c

    return "Other"

In [ ]:
# LOAD INPUTS
unresolved_df = read_table_file(PATH_UNRESOLVED)
rules_df = read_table_file(PATH_RULES)

print("unresolved_df:", unresolved_df.shape)
print("rules_df:", rules_df.shape)

display(unresolved_df.head(10))
display(rules_df.head(10))

unresolved_df: (693, 2)
rules_df: (16, 3)


,content,canonical_content
0,giao hữu bóng đá việt nam,giao hữu bóng đá việt nam
1,da bong hom nay,da bong hom nay
2,cinderella,cinderella
3,matter,matter
4,giao hữu bóng đá việt nam paris tăng,giao hữu bóng đá việt nam paris tăng
5,bi an sac dep,bi an sac dep
6,prime provider,prime provider
7,trực tiếp giao hữu,trực tiếp giao hữu
8,vtv6,vtv6
9,giao hữu việt nam,giao hữu việt nam


,alias,canonical_content,category
0,truc tiep bong da,trực tiếp bóng đá,Sport
1,bong da,bóng đá,Sport
2,u19 viet nam,u 19 việt nam,Sport
3,thu thach than tuong,thử thách thần tượng,Reality show
4,running man,running man,Reality show
5,vtv,vtv,News
6,viet nam,Việt Nam,Sport
7,bolero,bolero,Music
8,penthouse,penthouse,K-Drama
9,bat ma,bắt ma phá án,K-Drama


In [ ]:
# VALIDATE SCHEMA
required_unresolved_cols = {"content", "canonical_content"}
required_rules_cols = {"alias", "canonical_content", "category"}

missing_unresolved = required_unresolved_cols - set(unresolved_df.columns)
missing_rules = required_rules_cols - set(rules_df.columns)

if missing_unresolved:
    raise ValueError(f"Missing columns in unresolved_df: {missing_unresolved}")

if missing_rules:
    raise ValueError(f"Missing columns in rules_df: {missing_rules}")

print("Schema validation passed.")

Schema validation passed.


In [ ]:
# CLEAN INPUTS
unresolved_df = unresolved_df.copy()
unresolved_df["content"] = unresolved_df["content"].astype(str).str.strip()
unresolved_df["canonical_content"] = unresolved_df["canonical_content"].fillna(unresolved_df["content"])
unresolved_df["canonical_content"] = unresolved_df["canonical_content"].astype(str).str.strip()
unresolved_df["content_norm"] = unresolved_df["content"].apply(normalize_text)

rules_df = rules_df.copy()
rules_df["alias"] = rules_df["alias"].astype(str).str.strip()
rules_df["alias_norm"] = rules_df["alias"].apply(normalize_text)
rules_df["canonical_content"] = rules_df["canonical_content"].astype(str).str.strip()
rules_df["category"] = rules_df["category"].apply(normalize_category)

unresolved_df = unresolved_df[unresolved_df["content_norm"].notna()].copy()
rules_df = rules_df[rules_df["alias_norm"].notna()].copy()

print("Cleaned unresolved_df:", unresolved_df.shape)
print("Cleaned rules_df:", rules_df.shape)

Cleaned unresolved_df: (693, 3)
Cleaned rules_df: (16, 4)


In [ ]:
# EXACT RULE MATCH AGAIN
rules_map = {}
for _, row in rules_df.iterrows():
    key = row["alias_norm"]
    if key not in rules_map:
        rules_map[key] = {
            "canonical_content": row["canonical_content"],
            "category": normalize_category(row["category"])
        }

rule_matched_rows = []
still_unresolved_rows = []

for _, row in unresolved_df.iterrows():
    key = row["content_norm"]

    if key in rules_map:
        matched = rules_map[key]
        rule_matched_rows.append({
            "content": row["content"],
            "canonical_content": matched["canonical_content"],
            "category": matched["category"]
        })
    else:
        still_unresolved_rows.append({
            "content": row["content"],
            "canonical_content": row["canonical_content"]
        })

rule_matched_df = pd.DataFrame(rule_matched_rows)
still_unresolved_df = pd.DataFrame(still_unresolved_rows)

print("Matched by rules:", len(rule_matched_df))
print("Still unresolved:", len(still_unresolved_df))

display(rule_matched_df.head(10))
display(still_unresolved_df.head(10))

Matched by rules: 0
Still unresolved: 693


""


,content,canonical_content
0,giao hữu bóng đá việt nam,giao hữu bóng đá việt nam
1,da bong hom nay,da bong hom nay
2,cinderella,cinderella
3,matter,matter
4,giao hữu bóng đá việt nam paris tăng,giao hữu bóng đá việt nam paris tăng
5,bi an sac dep,bi an sac dep
6,prime provider,prime provider
7,trực tiếp giao hữu,trực tiếp giao hữu
8,vtv6,vtv6
9,giao hữu việt nam,giao hữu việt nam


In [ ]:
# LLM PROMPT
def build_prompt(batch_keywords):
    categories_str = ", ".join(FIXED_CATEGORIES)
    kw_list = "\n".join([f'{i+1}. "{kw}"' for i, kw in enumerate(batch_keywords)])

    return f"""
Bạn là chuyên gia phân loại nội dung cho nền tảng streaming video tại Việt Nam.

Danh sách category hợp lệ:
[{categories_str}]

Quy tắc:
- Chọn đúng 1 category hợp lệ nhất cho mỗi từ khóa
- Từ khóa có thể không dấu, sai dấu, viết tắt hoặc thiếu một phần tên
- Nếu là phim Trung nổi tiếng -> C-Drama
- Nếu là phim Hàn -> K-Drama
- Nếu là phim Việt -> V-Drama
- Nếu là phim Thái -> Thai-Drama
- Nếu thiên về hành động quốc tế -> Action
- Nếu là hoạt hình / anime / donghua -> Animation
- Nếu là âm nhạc / MV / ca sĩ -> Music
- Nếu là thể thao -> Sport
- Nếu là reality / gameshow -> Reality Show
- Nếu là phim tài liệu -> Documentary
- Nếu là đài truyền hình -> News
- Không xác định được -> Other

Yêu cầu output:
- Chỉ trả về JSON array
- Mỗi phần tử là object có đúng 2 field:
  - "content"
  - "category"
- Không markdown
- Không giải thích thêm
- Output phải cùng số lượng và cùng thứ tự với input

Ví dụ:
[
  {{"content": "ngự giao ký", "category": "C-Drama"}},
  {{"content": "the thing", "category": "Action"}}
]

Danh sách cần phân loại:
{kw_list}
"""

In [ ]:
# OLLAMA CALL
def call_ollama_batch(batch_keywords):
    prompt = build_prompt(batch_keywords)

    resp = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.05
            }
        },
        timeout=LLM_TIMEOUT
    )
    resp.raise_for_status()

    raw = resp.json()["response"].strip()
    raw = raw.replace("```json", "").replace("```", "").strip()

    data = json.loads(raw)

    results = []
    for i, kw in enumerate(batch_keywords):
        if i < len(data) and isinstance(data[i], dict):
            results.append({
                "content": kw,
                "category": normalize_category(data[i].get("category"))
            })
        else:
            results.append({
                "content": kw,
                "category": "Other"
            })

    return results

In [ ]:
# CHECKPOINT HELPERS
def load_checkpoint_jsonl(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        return pd.DataFrame(columns=["content", "canonical_content", "category"])

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    if not rows:
        return pd.DataFrame(columns=["content", "canonical_content", "category"])

    return pd.DataFrame(rows)

In [ ]:
# CLASSIFY WITH LLM
def classify_unresolved(unresolved_df: pd.DataFrame, batch_size: int = 20) -> pd.DataFrame:
    if unresolved_df.empty:
        return pd.DataFrame(columns=["content", "canonical_content", "category"])

    checkpoint_df = load_checkpoint_jsonl(PATH_CHECKPOINT)
    done_contents = set(checkpoint_df["content"].astype(str).tolist()) if not checkpoint_df.empty else set()

    work_df = unresolved_df[~unresolved_df["content"].astype(str).isin(done_contents)].copy()

    print("Pending LLM rows:", len(work_df))

    for start in range(0, len(work_df), batch_size):
        batch_df = work_df.iloc[start:start+batch_size].copy()
        batch_keywords = batch_df["content"].astype(str).tolist()

        print(f"Processing batch {start//batch_size + 1}/{math.ceil(len(work_df)/batch_size)}")

        try:
            preds = call_ollama_batch(batch_keywords)
        except Exception as e:
            print("Batch error:", e)
            preds = [{"content": x, "category": "Other"} for x in batch_keywords]

        batch_rows = []
        for i, pred in enumerate(preds):
            src = batch_df.iloc[i]
            row = {
                "content": src["content"],
                "canonical_content": src["canonical_content"],  # demo scope: keep canonical as prepared
                "category": pred["category"]
            }
            batch_rows.append(row)

        with open(PATH_CHECKPOINT, "a", encoding="utf-8") as f:
            for row in batch_rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    final_ckpt = load_checkpoint_jsonl(PATH_CHECKPOINT)
    final_ckpt = final_ckpt.drop_duplicates(subset=["content"], keep="last").reset_index(drop=True)
    return final_ckpt

In [ ]:
# RUN LLM LABELING
llm_labeled_df = classify_unresolved(still_unresolved_df, batch_size=LLM_BATCH_SIZE)

print("LLM labeled:", len(llm_labeled_df))
display(llm_labeled_df.head(20))

Pending LLM rows: 693
Processing batch 1/35
Processing batch 2/35
Processing batch 3/35
Processing batch 4/35
Processing batch 5/35
Processing batch 6/35
Processing batch 7/35
Processing batch 8/35
Processing batch 9/35
Processing batch 10/35
Processing batch 11/35
Processing batch 12/35
Processing batch 13/35
Processing batch 14/35
Processing batch 15/35
Processing batch 16/35
Processing batch 17/35
Processing batch 18/35
Processing batch 19/35
Processing batch 20/35
Processing batch 21/35
Processing batch 22/35
Processing batch 23/35
Processing batch 24/35
Processing batch 25/35
Processing batch 26/35
Processing batch 27/35
Processing batch 28/35
Processing batch 29/35
Processing batch 30/35
Processing batch 31/35
Processing batch 32/35
Processing batch 33/35
Processing batch 34/35
Processing batch 35/35
LLM labeled: 693


,content,canonical_content,category
0,giao hữu bóng đá việt nam,giao hữu bóng đá việt nam,Sport
1,da bong hom nay,da bong hom nay,Sport
2,cinderella,cinderella,Other
3,matter,matter,Other
4,giao hữu bóng đá việt nam paris tăng,giao hữu bóng đá việt nam paris tăng,Sport
5,bi an sac dep,bi an sac dep,Animation
6,prime provider,prime provider,Other
7,trực tiếp giao hữu,trực tiếp giao hữu,Sport
8,vtv6,vtv6,Other
9,giao hữu việt nam,giao hữu việt nam,Sport


In [ ]:
# COMBINE RULE MATCH + LLM MATCH
result_parts = []

if not rule_matched_df.empty:
    result_parts.append(rule_matched_df)

if not llm_labeled_df.empty:
    result_parts.append(llm_labeled_df)

if len(result_parts) == 0:
    final_new_labels = pd.DataFrame(columns=["content", "canonical_content", "category"])
else:
    final_new_labels = pd.concat(result_parts, ignore_index=True)

final_new_labels["category"] = final_new_labels["category"].apply(normalize_category)
final_new_labels["content"] = final_new_labels["content"].astype(str).str.strip()
final_new_labels["canonical_content"] = final_new_labels["canonical_content"].astype(str).str.strip()

final_new_labels = final_new_labels.drop_duplicates(subset=["content"], keep="first").reset_index(drop=True)

print("final_new_labels:", final_new_labels.shape)
display(final_new_labels.head(20))

final_new_labels: (693, 3)


,content,canonical_content,category
0,giao hữu bóng đá việt nam,giao hữu bóng đá việt nam,Sport
1,da bong hom nay,da bong hom nay,Sport
2,cinderella,cinderella,Other
3,matter,matter,Other
4,giao hữu bóng đá việt nam paris tăng,giao hữu bóng đá việt nam paris tăng,Sport
5,bi an sac dep,bi an sac dep,Animation
6,prime provider,prime provider,Other
7,trực tiếp giao hữu,trực tiếp giao hữu,Sport
8,vtv6,vtv6,Other
9,giao hữu việt nam,giao hữu việt nam,Sport


In [ ]:
# SAVE OUTPUT
final_new_labels.to_parquet(PATH_OUTPUT_PARQUET, index=False)
final_new_labels.to_csv(PATH_OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print("-", PATH_OUTPUT_PARQUET)
print("-", PATH_OUTPUT_CSV)

Saved:
- /content/customer360_output/dim_content_new_labels.parquet
- /content/customer360_output/dim_content_new_labels.csv
